# Platt Scaling & Isotonic Regression

Wiki reference for [Platt Scaling and Isotonic Regression](https://ml-viz-ruby.vercel.app/wiki/platt-scaling-and-isotonic-regression).

> **Copy to Drive** first (File → Save a copy in Drive) so your edits persist.

**The problem.** A trained classifier outputs a *score* — a logit, an SVM margin, a
forest vote fraction — that ranks examples well but does not read as a probability.
**Post-hoc calibration** learns a monotone map on a held-out set that turns those
scores into trustworthy probabilities, without retraining and without changing the
ranking. This notebook builds the two workhorse maps from scratch — **Platt scaling**
(a logistic fit) and **isotonic regression** (Pool-Adjacent-Violators) — then checks
them against scikit-learn.

## Setup

Imports and a dark plot style; a fixed seed keeps everything reproducible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['savefig.facecolor'] = '#0f1117'
rng = np.random.default_rng(7)

## A miscalibrated classifier to fix

We simulate an **overconfident** model: the true probability of the positive class
is a clean logistic in a latent variable `z`, but the model reports a *sharper*
sigmoid — `sigmoid(k*z)` with `k > 1` — the classic deep-net distortion that pushes
scores toward 0 and 1. Labels are drawn from the TRUE probability; we keep the
model's distorted (but correctly ranked) score.

In [ ]:
n = 4000
z = rng.normal(0, 1.5, n)                 # latent signal
p_true = 1 / (1 + np.exp(-z))             # TRUE P(y=1)
y = (rng.uniform(size=n) < p_true).astype(int)

# Model score: same ranking as p_true but SHARPER (k=2.5 > 1) => overconfident
score = 1 / (1 + np.exp(-2.5 * z))

# Split into calibration and test folds (NEVER calibrate on training data)
idx = rng.permutation(n)
cal, test = idx[: n // 2], idx[n // 2 :]
s_cal, y_cal = score[cal], y[cal]
s_test, y_test = score[test], y[test]
print('calibration points:', len(cal), '| test points:', len(test))

### Measure calibration: reliability + ECE

A helper that bins predictions by confidence and returns Expected Calibration Error
(the bin-weighted gap between confidence and accuracy). Lower is better.

In [ ]:
def reliability(p, y, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ids = np.clip(np.digitize(p, bins) - 1, 0, n_bins - 1)
    conf, acc, weight = [], [], []
    for b in range(n_bins):
        m = ids == b
        if m.sum() == 0:
            continue
        conf.append(p[m].mean()); acc.append(y[m].mean()); weight.append(m.mean())
    conf, acc, weight = map(np.array, (conf, acc, weight))
    ece = np.sum(weight * np.abs(acc - conf))
    return conf, acc, ece

_, _, ece_raw = reliability(s_test, y_test)
print(f'raw model ECE = {ece_raw:.4f}')

## 1 - Platt scaling from scratch

Fit $g(s) = 1/(1 + \exp(A\,s + B))$ by minimizing log loss on the calibration fold.
Two touches matter: (a) fit on the model *score* as the single feature, and (b) use
**regularized targets** from a Beta prior instead of raw 0/1 labels, so small folds
don't overfit to the extremes:

$$t_+ = \frac{N_+ + 1}{N_+ + 2}, \qquad t_- = \frac{1}{N_- + 2}$$

In [ ]:
def fit_platt(s, y):
    n_pos, n_neg = y.sum(), len(y) - y.sum()
    t = np.where(y == 1, (n_pos + 1) / (n_pos + 2), 1 / (n_neg + 2))  # smoothed targets

    def nll(params):
        A, B = params
        logits = A * s + B
        p = 1 / (1 + np.exp(logits))
        p = np.clip(p, 1e-12, 1 - 1e-12)
        return -np.mean(t * np.log(p) + (1 - t) * np.log(1 - p))

    res = minimize(nll, x0=[-1.0, 0.0], method='BFGS')
    return res.x  # A, B

A, B = fit_platt(s_cal, y_cal)
platt = lambda s: 1 / (1 + np.exp(A * s + B))
p_platt = platt(s_test)
_, _, ece_platt = reliability(p_platt, y_test)
print(f'A = {A:.3f}, B = {B:.3f}')
print(f'Platt ECE = {ece_platt:.4f}  (was {ece_raw:.4f})')

## 2 - Isotonic regression from scratch (PAV)

No sigmoid assumption: find the best **non-decreasing** step function by least
squares. The Pool-Adjacent-Violators algorithm does this exactly — sort by score,
then repeatedly pool any block whose value exceeds its right neighbour into a single
block holding their weighted mean.

In [ ]:
def pav(s, y):
    """Pool-Adjacent-Violators. Returns sorted scores + fitted monotone values."""
    order = np.argsort(s, kind='mergesort')
    xs, ys = s[order], y[order].astype(float)
    # each block: [sum, count]; value = sum / count
    vals = list(ys)
    wts = [1.0] * len(ys)
    i = 0
    while i < len(vals) - 1:
        if vals[i] > vals[i + 1] + 1e-15:            # violation -> pool
            new_w = wts[i] + wts[i + 1]
            new_v = (vals[i] * wts[i] + vals[i + 1] * wts[i + 1]) / new_w
            vals[i:i + 2] = [new_v]
            wts[i:i + 2] = [new_w]
            i = max(i - 1, 0)                        # back up to re-check left
        else:
            i += 1
    # expand blocks back to per-point fitted values
    fitted = np.concatenate([[v] * int(w) for v, w in zip(vals, wts)])
    return xs, fitted

# Quick check on the worked trace from the wiki:
demo_s = np.arange(6.0)
demo_y = np.array([0, 1, 0, 0, 1, 1])
_, demo_fit = pav(demo_s, demo_y)
print('worked PAV trace ->', np.round(demo_fit, 3))  # -> [0, .333, .333, .333, 1, 1]

Now fit PAV on the real calibration fold and interpolate to map any test score.
Isotonic is piecewise-constant, so we use `np.interp` on the block boundaries.

In [ ]:
xs_cal, fit_cal = pav(s_cal, y_cal)
iso = lambda s: np.interp(s, xs_cal, fit_cal)
p_iso = iso(s_test)
_, _, ece_iso = reliability(p_iso, y_test)
print(f'Isotonic ECE = {ece_iso:.4f}  (was {ece_raw:.4f})')

## 3 - The library way (and a cross-check)

scikit-learn ships both maps: `_SigmoidCalibration` is Platt, `IsotonicRegression`
is PAV. We fit them on the same calibration fold and confirm our from-scratch
probabilities match to a few decimals.

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import _SigmoidCalibration

sk_platt = _SigmoidCalibration().fit(s_cal, y_cal)
sk_iso = IsotonicRegression(out_of_bounds='clip').fit(s_cal, y_cal)

p_platt_sk = sk_platt.predict(s_test)
p_iso_sk = sk_iso.predict(s_test)

print('Platt   matches sklearn:', np.allclose(p_platt, p_platt_sk, atol=2e-2))
print('Isotonic matches sklearn:', np.allclose(p_iso, p_iso_sk, atol=2e-2))
print(f'sklearn Platt ECE    = {reliability(p_platt_sk, y_test)[2]:.4f}')
print(f'sklearn Isotonic ECE = {reliability(p_iso_sk, y_test)[2]:.4f}')

## 4 - Visualize the maps and the reliability gain

Left: the two calibration maps as functions of the raw score (diagonal = no change).
Right: reliability diagrams before and after — points on the dashed diagonal are
perfectly calibrated.

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.2))

grid = np.linspace(0, 1, 300)
axL.plot([0, 1], [0, 1], '--', color='#6b7280', label='identity')
axL.plot(grid, platt(grid), color='#6366f1', lw=2, label='Platt (sigmoid)')
axL.step(np.sort(s_cal), iso(np.sort(s_cal)), color='#14b8a6', lw=2, where='post', label='Isotonic (PAV)')
axL.set_xlabel('raw model score'); axL.set_ylabel('calibrated probability')
axL.set_title('Calibration maps'); axL.legend(); axL.set_xlim(0, 1); axL.set_ylim(0, 1)

axR.plot([0, 1], [0, 1], '--', color='#6b7280', label='perfect')
for p, name, col in [(s_test, 'raw', '#f43f5e'), (p_platt, 'Platt', '#6366f1'), (p_iso, 'Isotonic', '#14b8a6')]:
    conf, acc, ece = reliability(p, y_test)
    axR.plot(conf, acc, 'o-', color=col, label=f'{name} (ECE={ece:.3f})')
axR.set_xlabel('mean confidence'); axR.set_ylabel('empirical accuracy')
axR.set_title('Reliability diagram'); axR.legend(); axR.set_xlim(0, 1); axR.set_ylim(0, 1)
plt.tight_layout(); plt.show()

**What to notice.** The raw curve bows away from the diagonal (overconfident). Both
maps pull it back: Platt applies a smooth sigmoidal correction, while isotonic tracks
a monotone staircase that can bend to any shape — at the cost of small flat/jagged
steps where calibration data is thin. Neither reorders examples, so AUC is unchanged.

## 5 - Tradeoffs & when to use which

| | Platt scaling | Isotonic regression |
|---|---|---|
| Map family | logistic $1/(1+e^{As+B})$ | any monotone step function |
| Parameters | 2 ($A, B$) | non-parametric, $O(n)$ blocks |
| Assumption | sigmoidal distortion | monotonicity only |
| Data needed | little (hundreds) | more ($\gtrsim$ 1000) |
| Failure mode | can't fix non-sigmoidal shape | overfits on small data; flat/jagged |
| Cost | fit 2 params | $O(n)$ after an $O(n\log n)$ sort |

Both are **monotonic**, so neither changes ranking (AUC / top-1 accuracy preserved).
Reach for **Platt** on SVMs, boosted models, and small calibration folds; reach for
**isotonic** when you have plenty of calibration data and an oddly shaped reliability
curve. (For deep nets, one-parameter *temperature scaling* is often enough — see the
lesson.)

## ✏️ Your turn

**Task - Brier score before and after.** The Brier score is the mean squared error
between predicted probability and label, $\frac{1}{N}\sum (p_i - y_i)^2$ — a *proper*
scoring rule that rewards calibration. Compute it on the test fold for the raw scores
and for both calibrated versions, and confirm calibration lowers it.

Fill in the `# TODO(you)` line; the assert passes silently when correct.

In [ ]:
def brier(p, y):
    # TODO(you): return the mean squared error between p and y
    ...

# --- checks (do not edit) ---
b_raw = brier(s_test, y_test)
b_platt = brier(p_platt, y_test)
b_iso = brier(p_iso, y_test)
assert abs(b_raw - np.mean((s_test - y_test) ** 2)) < 1e-12, 'brier() is not MSE'
assert b_platt < b_raw and b_iso < b_raw, 'calibration should reduce the Brier score'
print(f'Brier  raw={b_raw:.4f}  Platt={b_platt:.4f}  Isotonic={b_iso:.4f}')

<details><summary>Solution</summary>

```python
def brier(p, y):
    return np.mean((p - y) ** 2)
```

Both calibrated versions score lower than the raw model because the overconfident
raw scores are penalized by the squared term. Brier decomposes into calibration +
refinement, so fixing calibration while preserving ranking can only help.
</details>

## Key takeaways

- **Post-hoc calibration** maps trained-model scores to probabilities on a *held-out*
  fold, without retraining and without changing ranking (AUC/accuracy preserved).
- **Platt scaling** = 1-D logistic on the score with Beta-smoothed targets; two
  parameters, robust on small folds, assumes a *sigmoidal* distortion.
- **Isotonic regression** = best monotone step function via **PAV**; non-parametric
  and flexible, but needs more data or it overfits into a jagged staircase.
- Both are monotonic; pick Platt for small folds / SVMs / boosting, isotonic for
  large folds with odd reliability curves, temperature scaling for deep nets.
- Next: the [Calibration & Uncertainty lesson](https://ml-viz-ruby.vercel.app/courses/model-evaluation/07-calibration-and-uncertainty).